In [ ]:
import pandas as pd

test_df= pd.read_excel('test_data.xlsx')

label_list=['hotel-general', 'bathroom-general', 'hotel-atmosphere', 'bathroom-atmosphere',
    'hotel-cleanliness', 'bathroom-cleanliness', 'hotel-facilities', 'bathroom-facilities',
    'hotel-location', 'bed-general', 'hotel-price', 'bed-cleanliness',
    'room-general', 'catering-general', 'room-atmosphere', 'catering-price',
    'room-cleanliness', 'parking', 'room-facilities', 'staff']

In [ ]:
import re

def clean_and_header(text):
    if not isinstance(text, str):
        return ''
    header=''

    if '·' in text:
        parts= text.split('·', 1)
        header= parts[0].strip().lower()
        text= parts[1].strip()

        if 'disliked' in header:
            header= "[DISLIKED] "
        elif 'liked' in header:
            header= "[LIKED] "
        else:
            return text.replace('·', '').strip()
        text= text

    text= f"{header}{text}"
    text= re.sub(r'http\S+', '', text)
    text= re.sub(r'[^\w\s\-àáâãäåæçèéêëìíîïðñòóôõöùúûüý·\[\]]', ' ', text)
    text= re.sub(r'\s+', ' ', text).strip()

    return text

text_col= [c for c in test_df.columns if c not in label_list][0]
clean_texts= test_df[text_col].apply(clean_and_header)

In [ ]:
from torch.utils.data import Dataset

class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts= [str(t) for t in texts]
        self.tokenizer= tokenizer
        self.max_length= max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        encoding= self.tokenizer(
            self.texts[idx],
            add_special_tokens= True,
            max_length= self.max_length,
            padding='max_length',
            truncation= True,
            return_tensors= 'pt'
        )
        return{
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0)
        }

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np

device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MODEL_PATH= 'ainniy/xlm-roberta-hotel-topics'
THRESHOLD_PATH= 'best_thresholds.npy'

tokenizer= AutoTokenizer.from_pretrained(MODEL_PATH)
model= AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

thresholds= np.load(THRESHOLD_PATH)

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm

test_dataset= InferenceDataset(clean_texts, tokenizer, max_length=128)
test_loader= DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

all_probs= []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids= batch['input_ids'].to(device)
        attention_mask= batch['attention_mask'].to(device)

        outputs= model(input_ids, attention_mask=attention_mask)
        probs= torch.sigmoid(outputs.logits).cpu().numpy()
        all_probs.append(probs)

all_probs= np.concatenate(all_probs, axis=0)

predictions= (all_probs >= thresholds).astype(int)

In [ ]:
output_df= test_df.copy()

for i, label in enumerate(label_list):
    output_df[label]= predictions[:, i]

output_df.to_excel('output.xlsx', index=False)